# LeetCode 1794 — Count Pairs of Equal Substrings With Minimum Difference bằng Python

Link đề: https://leetcode.com/problems/count-pairs-of-equal-substrings-with-minimum-difference/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Greedy

> **Ghi chú (bài Premium 🔒):** đề trong notebook này được viết lại theo hiểu biết về bài và theo bản mô tả công khai của kho doocs/leetcode. Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def countQuadruples(self, firstString: str, secondString: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Đọc kỹ một điều kiện "lạ": đếm các bộ bốn `(i, j, a, b)` có chuỗi con bằng nhau **và** đạt giá trị nhỏ nhất của `j − a`.
2. Chứng minh insight then chốt: bộ tối ưu luôn là chuỗi con **độ dài 1** (`i = j`, `a = b`).
3. Từ vét cạn `O(n²m²)` → duyệt cặp ký tự `O(n·m)` → **vị trí xuất hiện cuối** trong chuỗi thứ hai `O(n + m)` → chỉ 26 ký tự `O(n + m + 26)`.
4. Đếm cẩn thận khi có **hòa** giá trị nhỏ nhất (cập nhật "đặt lại về 1" hay "cộng thêm 1").
5. Kiểm thử đối chiếu với vét cạn đầy đủ định nghĩa trên chuỗi ngắn.

## 2. Đề bài & ví dụ

Cho hai chuỗi `firstString` (độ dài `n`) và `secondString` (độ dài `m`), chỉ gồm chữ thường. Đếm số bộ chỉ số `(i, j, a, b)` thỏa:
- `0 ≤ i ≤ j < n` và `0 ≤ a ≤ b < m`;
- `firstString[i..j] == secondString[a..b]` (bao gồm hai đầu);
- `j − a` là **nhỏ nhất có thể** trong mọi bộ thỏa hai điều kiện trên.

**Ví dụ 1:** `firstString = "abcd"`, `secondString = "bccda"` → `1`: bộ duy nhất là `(0, 0, 4, 4)` (ký tự `a`), `j − a = −4`.

**Ví dụ 2:** `firstString = "ab"`, `secondString = "cd"` → `0` (không có chuỗi con chung).

**Ràng buộc:** `1 ≤ n, m ≤ 2·10⁵`.

In [1]:
import random
import time
from typing import List

examples = [("abcd", "bccda", 1), ("ab", "cd", 0)]
for f, s, exp in examples:
    print(f"first={f!r}, second={s!r} -> kỳ vọng {exp}")

first='abcd', second='bccda' -> kỳ vọng 1
first='ab', second='cd' -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Chỉ số trong chuỗi con

`s[i..j]` (bao gồm `j`) trong Python là `s[i:j+1]`. Bộ bốn chọn **một** chuỗi con của mỗi chuỗi.

### 3.2 Từ điển "vị trí xuất hiện cuối"

`{c: i for i, c in enumerate(s)}` — vì gán đè khi duyệt, mỗi ký tự giữ chỉ số **cuối cùng**.

In [2]:
s = "bccda"
last = {c: i for i, c in enumerate(s)}
first = {}
for i, c in enumerate(s):
    first.setdefault(c, i)
print("cuối:", last, "| đầu:", first)

cuối: {'b': 0, 'c': 2, 'd': 3, 'a': 4} | đầu: {'b': 0, 'c': 1, 'd': 3, 'a': 4}


### 3.3 Đếm phần tử đạt giá trị nhỏ nhất trong một lượt

Giữ `best` và `cnt`: gặp giá trị nhỏ hơn ⇒ `best = v, cnt = 1`; bằng ⇒ `cnt += 1`; lớn hơn ⇒ bỏ qua.

In [3]:
best, cnt = float("inf"), 0
for v in [3, 1, 4, 1, 5, 1]:
    if v < best: best, cnt = v, 1
    elif v == best: cnt += 1
print(best, cnt)

1 3


### 3.4 Mảng 26 phần tử thay cho dict

`idx = ord(c) − 97`; giá trị mặc định `−1` nghĩa là "không xuất hiện".

## 4. Ý tưởng nền tảng / trực giác

**Quan sát chính:** nếu `(i, j, a, b)` có `first[i..j] == second[a..b]` thì **ký tự đầu** cũng bằng nhau: `first[i] == second[a]` ⇒ `(i, i, a, a)` cũng hợp lệ, với `i − a ≤ j − a`. Vậy giá trị nhỏ nhất của `j − a` luôn đạt được bởi một bộ độ dài 1. Hơn nữa, nếu chuỗi con dài `L ≥ 2` thì `(i, i, a, a)` cho giá trị **nhỏ hơn hẳn** (`i < j`) ⇒ bộ dài `≥ 2` **không bao giờ** đạt min.

⇒ Chỉ cần đếm các cặp `(j, a)` với `first[j] == second[a]` có `j − a` nhỏ nhất.

Với `j` cố định, `j − a` nhỏ nhất khi `a` **lớn nhất** ⇒ `a = last[first[j]]` trong `second`.

**Ví dụ 1:** `last` của `"bccda"` = `{b:0, c:2, d:3, a:4}`. Với `"abcd"`: `j=0 'a': 0−4 = −4`; `j=1 'b': 1−0 = 1`; `j=2 'c': 2−2 = 0`; `j=3 'd': 3−3 = 0` ⇒ min `−4`, đạt **1** lần.

In [4]:
f, s, _ = examples[0]
last = {c: i for i, c in enumerate(s)}
print([(j, c, j - last[c]) for j, c in enumerate(f) if c in last])

[(0, 'a', -4), (1, 'b', 1), (2, 'c', 0), (3, 'd', 0)]


## 5. Cách 1 — Brute Force / baseline: đúng định nghĩa, mọi bộ bốn

In [5]:
def count_bruteforce(first, second):
    n, m = len(first), len(second)
    best, cnt = None, 0
    for i in range(n):
        for j in range(i, n):
            L = j - i + 1
            sub = first[i:j + 1]
            for a in range(m - L + 1):
                if second[a:a + L] == sub:
                    v = j - a
                    if best is None or v < best: best, cnt = v, 1
                    elif v == best: cnt += 1
    return cnt

for f, s, exp in examples:
    got = count_bruteforce(f, s); print(got, "kỳ vọng", exp); assert got == exp

1 kỳ vọng 1
0 kỳ vọng 0


**Độ phức tạp:** `O(n² · m · L)` ~ `O(n³m)` — chỉ dùng cho chuỗi rất ngắn.

**Nút thắt:** xét cả chuỗi con dài, trong khi chúng không bao giờ là tối ưu.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề (độ dài 1):** mọi bộ đạt min đều có `i = j` và `a = b`; và với mọi cặp `(j, a)` có `first[j] == second[a]`, bộ `(j, j, a, a)` hợp lệ. Vì vậy:

`đáp án = #{(j, a) : first[j] == second[a] và j − a = min}`.

**Rút gọn hơn nữa:** với mỗi `j`, chỉ `a = last[first[j]]` có thể đạt min (mọi `a` khác của cùng ký tự cho `j − a` lớn hơn hẳn). Với mỗi ký tự `c`, chỉ `j = firstIdx[c]` (lần xuất hiện **đầu** trong `firstString`) có thể đạt min. ⇒ Mỗi ký tự đóng góp **tối đa một** ứng viên `firstIdx[c] − lastIdx[c]`.

**Quy tắc đếm:** duyệt ứng viên, gặp nhỏ hơn ⇒ đặt lại đếm = 1; bằng ⇒ cộng 1.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Duyệt mọi cặp vị trí có ký tự bằng nhau

In [6]:
def count_pairs(first, second):
    best, cnt = None, 0
    for j, c in enumerate(first):
        for a, d in enumerate(second):
            if c == d:
                v = j - a
                if best is None or v < best: best, cnt = v, 1
                elif v == best: cnt += 1
    return cnt

print([count_pairs(f, s) for f, s, _ in examples])

[1, 0]


**Độ phức tạp Cách 2:** `O(n · m)` = `4·10¹⁰` tệ nhất ⇒ TLE; nhưng đã đúng nhờ bổ đề độ dài 1.

### 7.2 Cách 3 — Vị trí cuối trong `secondString`

In [7]:
def count_last(first, second):
    last = {c: i for i, c in enumerate(second)}
    best, cnt = float("inf"), 0
    for j, c in enumerate(first):
        if c in last:
            v = j - last[c]
            if v < best: best, cnt = v, 1
            elif v == best: cnt += 1
    return cnt

print([count_last(f, s) for f, s, _ in examples])

[1, 0]


**Code nộp LeetCode (Cách 3):** xem mục 9. **Độ phức tạp:** Time `O(n + m)`, Space `O(26)`.

### 7.3 Cách 4 — Chỉ 26 ứng viên: `firstIdx[c] − lastIdx[c]`

In [8]:
class Solution26:
    def countQuadruples(self, firstString: str, secondString: str) -> int:
        fi = [-1] * 26; la = [-1] * 26
        for j in range(len(firstString) - 1, -1, -1):       # duyệt ngược để giữ lần xuất hiện đầu
            fi[ord(firstString[j]) - 97] = j
        for a, c in enumerate(secondString):
            la[ord(c) - 97] = a
        cands = [fi[k] - la[k] for k in range(26) if fi[k] != -1 and la[k] != -1]
        if not cands:
            return 0
        mn = min(cands)
        return cands.count(mn)

print([Solution26().countQuadruples(f, s) for f, s, _ in examples])

[1, 0]


**Độ phức tạp Cách 4:** `O(n + m + 26)`. Ưu điểm: thấy rõ đáp án `≤ 26`.

**Vì sao mỗi ký tự góp tối đa 1:** với ký tự `c`, giá trị `j − a` nhỏ nhất là `firstIdx[c] − lastIdx[c]`, đạt **duy nhất** tại `(j, a) = (firstIdx[c], lastIdx[c])`. Hai ký tự khác nhau có thể cùng đạt min ⇒ đếm số ký tự hòa.

## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 1** và một ví dụ tự dựng có **hòa** (`first = "ab"`, `second = "ab"` — đối chiếu brute force; nên đối chiếu lại đề gốc):

In [9]:
def trace_last(first, second):
    last = {c: i for i, c in enumerate(second)}
    print("last =", last)
    best, cnt = float("inf"), 0
    for j, c in enumerate(first):
        if c not in last:
            print(f"  j={j} {c!r}: không có trong second"); continue
        v = j - last[c]
        if v < best:
            best, cnt = v, 1; act = "nhỏ hơn -> đặt lại cnt=1"
        elif v == best:
            cnt += 1; act = "bằng -> cnt+=1"
        else:
            act = "lớn hơn -> bỏ"
        print(f"  j={j} {c!r}: a=last={last[c]}, j-a={v} -> {act} | best={best}, cnt={cnt}")
    return cnt

assert trace_last("abcd", "bccda") == 1
print()
assert trace_last("ab", "ab") == count_bruteforce("ab", "ab") == 2

last = {'b': 0, 'c': 2, 'd': 3, 'a': 4}
  j=0 'a': a=last=4, j-a=-4 -> nhỏ hơn -> đặt lại cnt=1 | best=-4, cnt=1
  j=1 'b': a=last=0, j-a=1 -> lớn hơn -> bỏ | best=-4, cnt=1
  j=2 'c': a=last=2, j-a=0 -> lớn hơn -> bỏ | best=-4, cnt=1
  j=3 'd': a=last=3, j-a=0 -> lớn hơn -> bỏ | best=-4, cnt=1

last = {'a': 0, 'b': 1}
  j=0 'a': a=last=0, j-a=0 -> nhỏ hơn -> đặt lại cnt=1 | best=0, cnt=1
  j=1 'b': a=last=1, j-a=0 -> bằng -> cnt+=1 | best=0, cnt=2


| j | ký tự | a = last | j − a | best | cnt |
|---|---|---|---|---|---|
| 0 | a | 4 | −4 | −4 | 1 |
| 1 | b | 0 | 1 | −4 | 1 |
| 2 | c | 2 | 0 | −4 | 1 |
| 3 | d | 3 | 0 | −4 | **1** |

Với `("ab", "ab")`: `a: 0 − 0 = 0`, `b: 1 − 1 = 0` ⇒ hòa ⇒ `2` (bộ `(0,0,0,0)` và `(1,1,1,1)`).

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — vị trí cuối trong `secondString`.

In [10]:
class Solution:
    def countQuadruples(self, firstString: str, secondString: str) -> int:
        last = {c: i for i, c in enumerate(secondString)}
        best, cnt = float("inf"), 0
        for j, c in enumerate(firstString):
            if c in last:
                v = j - last[c]
                if v < best:
                    best, cnt = v, 1
                elif v == best:
                    cnt += 1
        return cnt

for f, s, exp in examples:
    got = Solution().countQuadruples(f, s); print(got, exp); assert got == exp

1 1
0 0


## 10. Vì sao thuật toán đúng

Gọi `Q` = tập các bộ hợp lệ (chuỗi con bằng nhau), `M = min_{Q} (j − a)`.

**Bổ đề 1.** Nếu `(i, j, a, b) ∈ Q` thì `(i, i, a, a) ∈ Q` và `i − a ≤ j − a`, dấu bằng ⇔ `i = j`. *Lý do:* hai chuỗi con bằng nhau thì ký tự đầu bằng nhau; `i ≤ j`.

**Hệ quả.** `M` = min trên các bộ độ dài 1; và mọi bộ đạt `M` phải có `i = j` (nếu không, bộ độ dài 1 tương ứng cho giá trị nhỏ hơn hẳn — mâu thuẫn). Bộ có `i = j` thì `b − a = j − i = 0` ⇒ `a = b`. Vậy bộ đạt min ⇔ cặp `(j, a)` với `first[j] == second[a]`, `j − a = M`.

**Bổ đề 2.** Với `j` cố định, các `a` hợp lệ là vị trí của ký tự `first[j]` trong `second`; `j − a` nhỏ nhất **duy nhất** tại `a = last[first[j]]`. Vậy mỗi `j` có tối đa một cặp đạt min, và cặp đó là `(j, last[first[j]])`.

Cách 3 duyệt đúng các ứng viên này và đếm số ứng viên bằng min ⇒ đúng. ∎

## 11. Phân tích Time / Space Complexity

Gọi `n = len(firstString)`, `m = len(secondString)` (`≤ 2·10⁵`).

| Cách | Time | Space |
|---|---|---|
| 1. Mọi bộ bốn | `O(n³ m)` | `O(n)` |
| 2. Mọi cặp ký tự | `O(n m)` | `O(1)` |
| 3. Vị trí cuối | `O(n + m)` | `O(26)` |
| 4. 26 ứng viên | `O(n + m + 26)` | `O(26)` |

In [11]:
rng = random.Random(1794)
A = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(2 * 10 ** 5))
B = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(2 * 10 ** 5))
for name, f in [("Cách 3", Solution().countQuadruples), ("Cách 4", Solution26().countQuadruples)]:
    t = time.perf_counter(); r = f(A, B); print(f"{name}: {r} ({(time.perf_counter()-t)*1000:.0f} ms)")
t = time.perf_counter(); count_pairs(A[:1000], B[:1000]); print(f"Cách 2 trên 1000x1000: {(time.perf_counter()-t)*1000:.0f} ms (2·10^5 sẽ chậm hơn 4·10^4 lần)")

Cách 3: 1 (19 ms)
Cách 4: 1 (11 ms)
Cách 2 trên 1000x1000: 25 ms (2·10^5 sẽ chậm hơn 4·10^4 lần)


## 12. Bộ test

In [12]:
ALL = {"pairs": count_pairs, "last": count_last, "Sol26": Solution26().countQuadruples, "submit": Solution().countQuadruples}
def run_all(f, s, with_brute=True):
    res = {k: g(f, s) for k, g in ALL.items()}
    if with_brute:
        res["brute"] = count_bruteforce(f, s)
    assert len(set(res.values())) == 1, (f, s, res)
    return next(iter(res.values()))

for f, s, exp in examples:
    assert run_all(f, s) == exp
edge = [("a", "a", 1), ("a", "b", 0), ("ab", "ab", 2), ("aa", "aa", 1), ("abc", "cba", 1), ("ba", "ab", 1), ("abc", "abc", 3)]
for f, s, exp in edge:
    assert run_all(f, s) == exp, (f, s)
rng = random.Random(17940)
for _ in range(1500):
    alpha = rng.choice(["ab", "abc", "abcdef"])
    f = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 8)))
    s = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 8)))
    run_all(f, s)
print("Ví dụ, edge case (có hòa), 1500 test vét cạn đầy đủ: OK")

Ví dụ, edge case (có hòa), 1500 test vét cạn đầy đủ: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng vị trí xuất hiện **đầu** trong `secondString`

In [13]:
def wrong_first_occ(first, second):
    pos = {}
    for i, c in enumerate(second):
        pos.setdefault(c, i)                             # SAI: cần vị trí CUỐI để j - a nhỏ nhất
    best, cnt = float("inf"), 0
    for j, c in enumerate(first):
        if c in pos:
            v = j - pos[c]
            if v < best: best, cnt = v, 1
            elif v == best: cnt += 1
    return cnt

f, s = "ab", "bab"                                  # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_first_occ(f, s), "| đúng:", Solution().countQuadruples(f, s), "| brute:", count_bruteforce(f, s))

sai : 1 | đúng: 2 | brute: 2


**Vì sao sai:** muốn `j − a` nhỏ thì `a` phải **lớn** ⇒ vị trí cuối. Với `"bab"`, `b` cuối ở chỉ số 2 cho `1 − 2 = −1` (cùng `a: 0 − 1 = −1` ⇒ 2 bộ hòa); dùng vị trí đầu cho `b: 1 − 0 = 1` ⇒ chỉ đếm 1. **Sửa:** `last = {c: i for i, c in enumerate(second)}`.

### Lỗi 2 — Quên đặt lại bộ đếm khi gặp giá trị nhỏ hơn

In [14]:
def wrong_no_reset(first, second):
    last = {c: i for i, c in enumerate(second)}
    best, cnt = float("inf"), 0
    for j, c in enumerate(first):
        if c in last:
            v = j - last[c]
            if v < best: best = v; cnt += 1                  # SAI: phải cnt = 1
            elif v == best: cnt += 1
    return cnt

f, s = "ab", "acb"                                 # ví dụ tự dựng (đối chiếu lại đề gốc): giá trị 0 rồi -1
print("sai :", wrong_no_reset(f, s), "| đúng:", Solution().countQuadruples(f, s), "| brute:", count_bruteforce(f, s))

sai : 2 | đúng: 1 | brute: 1


**Vì sao sai:** mỗi lần tìm được min mới, các bộ đạt min cũ không còn là tối ưu. Ở đây `a` cho `0 − 0 = 0` rồi `b` cho `1 − 2 = −1` nhỏ hơn: đáp án là 1 bộ, code sai đếm thành 2. **Sửa:** `cnt = 1` khi `v < best`.

### Lỗi 3 — Đếm mọi cặp ký tự bằng nhau (bỏ điều kiện "nhỏ nhất")

In [15]:
def wrong_count_all(first, second):
    return sum(1 for c in first for d in second if c == d)

f, s, exp = examples[0]
print("sai :", wrong_count_all(f, s), "| đúng:", exp)

sai : 5 | đúng: 1


**Vì sao sai:** chỉ đếm các bộ đạt **giá trị nhỏ nhất** của `j − a`. **Sửa:** theo dõi `best`, `cnt`.

### Lỗi 4 — Nghĩ rằng phải xét chuỗi con dài (và cài phức tạp không cần thiết)

Đây không phải lỗi sai kết quả mà là lỗi **hiểu đề**: nhiều bạn viết so khớp chuỗi con (hashing, Z-function…) — tốn công và dễ sai. Bổ đề ở mục 10 cho thấy chuỗi con dài **không bao giờ** đạt min. Kiểm chứng bằng vét cạn: trong mọi bộ đạt min, `i == j`.

In [16]:
def optimal_quads(first, second):
    n, m = len(first), len(second); best, out = None, []
    for i in range(n):
        for j in range(i, n):
            L = j - i + 1
            for a in range(m - L + 1):
                if second[a:a + L] == first[i:j + 1]:
                    v = j - a
                    if best is None or v < best: best, out = v, [(i, j, a, a + L - 1)]
                    elif v == best: out.append((i, j, a, a + L - 1))
    return out

rng = random.Random(3)
ok = True
for _ in range(500):
    f = "".join(rng.choice("ab") for _ in range(rng.randint(1, 6))); s = "".join(rng.choice("ab") for _ in range(rng.randint(1, 6)))
    ok &= all(i == j and a == b for i, j, a, b in optimal_quads(f, s))
print("Mọi bộ tối ưu đều có độ dài 1:", ok, "| ví dụ:", optimal_quads("abab", "baba"))

Mọi bộ tối ưu đều có độ dài 1: True | ví dụ: [(0, 0, 3, 3)]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có điều kiện "giá trị nhỏ nhất của một biểu thức chỉ số" ⇒ thử **thu nhỏ** đối tượng (chuỗi con → một ký tự) xem có làm biểu thức tốt hơn không.
- "Min `j − a`" ⇒ `j` nhỏ, `a` lớn ⇒ vị trí **đầu** trong chuỗi thứ nhất, **cuối** trong chuỗi thứ hai.
- Đếm số lần đạt min: mẫu `best/cnt` với đặt lại.
- Bảng chữ 26 ⇒ đáp án bị chặn bởi 26.

## 15. Biến thể của bài

1. **Tối đa `j − a`**: dùng vị trí **cuối** trong `firstString` cho `j`, vị trí **đầu** trong `secondString` cho `a`… nhưng cẩn thận: với max, chuỗi con dài **lại có lợi** (`j` tăng) — bài khó hơn hẳn (Bài tập 5 dạng đơn giản: chỉ xét độ dài 1).
2. **Min `|j − a|`**: cần vị trí gần nhất ⇒ nhị phân trên danh sách vị trí.
3. **Đếm bộ với chuỗi con độ dài cố định `L`**: hashing các chuỗi con độ dài `L`.
4. **1638 Count Substrings That Differ by One Character**: đếm cặp chuỗi con, khác một ký tự.
5. **Trả về chính các bộ tối ưu** (Bài tập 4).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi bộ bốn | Đúng định nghĩa | `O(n³m)` | `O(n)` | Kiểm chứng |
| 2. Mọi cặp ký tự | Bổ đề độ dài 1 | `O(nm)` | `O(1)` | Trung gian |
| 3. Vị trí cuối | `a = last[c]` | `O(n + m)` | `O(26)` | **Bản nộp** |
| 4. 26 ứng viên | `first[c] − last[c]` | `O(n + m)` | `O(26)` | Gọn, thấy đáp án ≤ 26 |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `("abc", "xyz")`; (b) `("aaa", "a")`; (c) `("xyz", "zyx")`; (d) `("ab", "ba")`.

**Bài 2 (tính tay):** Với `first = "acb"`, `second = "bca"`, lập bảng `j, ký tự, last, j − a` và đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `last_index(s)` trả mảng 26 phần tử vị trí cuối (−1 nếu không có).

**Bài 4 (tự viết lại):** Viết `optimal_list(first, second)` trả **danh sách** các bộ `(i, j, a, b)` đạt min trong `O(n + m)`.

**Bài 5 (mini challenge):** Viết `count_max_len1(first, second)` đếm các cặp `(j, a)` với `first[j] == second[a]` đạt **lớn nhất** `j − a` (chỉ xét độ dài 1), `O(n + m)`, đối chiếu vét cạn.

In [17]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def last_index_todo(s):
    res = [-1] * 26
    # TODO: for i, c in enumerate(s): res[ord(c) - 97] = i
    return res

In [18]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def optimal_list(first, second):
    # TODO: như Cách 3 nhưng lưu (j, j, a, a) thay vì chỉ đếm
    pass

def count_max_len1(first, second):
    # TODO: vị trí CUỐI trong first, vị trí ĐẦU trong second, 26 ứng viên, đếm số đạt max
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tìm `last`. Mức 2: tính `j − last[c]`. Mức 3: (a) `0`; (b) `1` (`a: 0 − 0 = 0` nhỏ nhất ở `j = 0`); (c) `1` (`x: 0 − 2 = −2`); (d) `1` (`a: 0 − 1 = −1`, `b: 1 − 0 = 1`).

**Bài 2.** Mức 1: `last = {b:0, c:1, a:2}`. Mức 2: `a: 0−2 = −2`, `c: 1−1 = 0`, `b: 2−0 = 2`. Mức 3: min `−2`, đáp án `1`.

**Bài 3.** Mức 1: mảng 26. Mức 2: ghi đè theo thứ tự duyệt. Mức 3: so với `dict` của Cách 3.

**Bài 4.** Mức 1: biến `best`, list `out`. Mức 2: nhỏ hơn ⇒ `out = [bộ]`; bằng ⇒ `append`. Mức 3: so với `optimal_quads` (vét cạn) đã sắp.

**Bài 5.** Mức 1: max `j − a` ⇒ `j` lớn, `a` nhỏ. Mức 2: mỗi ký tự một ứng viên `lastFirst[c] − firstSecond[c]`. Mức 3: đếm số ký tự đạt max.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
b1 = [Solution().countQuadruples(f, s) for f, s in [("abc", "xyz"), ("aaa", "a"), ("xyz", "zyx"), ("ab", "ba")]]
print("Bài 1:", b1); assert b1 == [0, 1, 1, 1]
print("Bài 2:"); assert trace_last("acb", "bca") == 1
def last_index_todo(s):
    res = [-1] * 26
    for i, c in enumerate(s):
        res[ord(c) - 97] = i
    return res
assert last_index_todo("bccda")[:5] == [4, 0, 2, 3, -1]
print("Bài 3:", last_index_todo("bccda")[:5])

Bài 1: [0, 1, 1, 1]
Bài 2:
last = {'b': 0, 'c': 1, 'a': 2}
  j=0 'a': a=last=2, j-a=-2 -> nhỏ hơn -> đặt lại cnt=1 | best=-2, cnt=1
  j=1 'c': a=last=1, j-a=0 -> lớn hơn -> bỏ | best=-2, cnt=1
  j=2 'b': a=last=0, j-a=2 -> lớn hơn -> bỏ | best=-2, cnt=1
Bài 3: [4, 0, 2, 3, -1]


In [20]:
# Bài 4, 5
def optimal_list(first, second):
    last = {c: i for i, c in enumerate(second)}
    best, out = float("inf"), []
    for j, c in enumerate(first):
        if c in last:
            v = j - last[c]
            if v < best: best, out = v, [(j, j, last[c], last[c])]
            elif v == best: out.append((j, j, last[c], last[c]))
    return out

def count_max_len1(first, second):
    lf = {c: i for i, c in enumerate(first)}
    fs = {}
    for i, c in enumerate(second):
        fs.setdefault(c, i)
    cands = [lf[c] - fs[c] for c in lf if c in fs]
    return cands.count(max(cands)) if cands else 0

def max_brute(first, second):
    vals = [j - a for j, c in enumerate(first) for a, d in enumerate(second) if c == d]
    return vals.count(max(vals)) if vals else 0

rng = random.Random(8)
for _ in range(800):
    f = "".join(rng.choice("abc") for _ in range(rng.randint(1, 8))); s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 8)))
    assert sorted(optimal_list(f, s)) == sorted(optimal_quads(f, s))
    assert count_max_len1(f, s) == max_brute(f, s)
print("Bài 4, 5: OK |", optimal_list("abab", "baba"), count_max_len1("abcd", "bccda"))

Bài 4, 5: OK | [(0, 0, 3, 3)] 2


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
last = {c: i for i, c in enumerate(secondString)}
best, cnt = inf, 0
for j, c in enumerate(firstString):
    if c in last:
        v = j - last[c]
        if v < best: best, cnt = v, 1
        elif v == best: cnt += 1
return cnt
```

**Checklist:**
- [ ] Hiểu vì sao chỉ xét chuỗi con **độ dài 1**?
- [ ] Dùng vị trí **cuối** trong `secondString`?
- [ ] Đặt lại `cnt = 1` khi gặp min mới; cộng khi hòa?
- [ ] Trả `0` khi không có ký tự chung?
- [ ] Đã test hòa giữa nhiều ký tự, chuỗi giống nhau, không có ký tự chung?